# 03 — Train Orthoptera Classifier
Fine-tune a CNN on the prepared label splits using OpenSoundscape, then export the trained model to `models/` for use in BASE.

**Kernel:** `Python (orthoptera-training)`  
**Prerequisites:** Run `02_prepare_labels.ipynb` first to generate `training/data/train.csv` etc.

In [ ]:
import pandas as pd
import os
from opensoundscape import CNN

DATA_DIR   = "../../training/data"
MODELS_DIR = "../../models"
os.makedirs(MODELS_DIR, exist_ok=True)

train = pd.read_csv(os.path.join(DATA_DIR, "train.csv"), index_col=[0, 1, 2])
val   = pd.read_csv(os.path.join(DATA_DIR, "val.csv"),   index_col=[0, 1, 2])
test  = pd.read_csv(os.path.join(DATA_DIR, "test.csv"),  index_col=[0, 1, 2])

print(f"Classes: {list(train.columns)}")
print(f"Train: {len(train)}  Val: {len(val)}  Test: {len(test)}")

In [ ]:
# ── Build model ───────────────────────────────────────────────────────────────
# ResNet18 is a good starting point — fast to train, works well with limited data.
# Swap for 'resnet50' or 'efficientnet_b0' if you have more clips and GPU time.

model = CNN(
    architecture='resnet18',
    classes=list(train.columns),
    sample_duration=3.0,    # seconds per clip — matches BASE's chunk_duration
)

In [ ]:
# ── Train ─────────────────────────────────────────────────────────────────────
# epochs=30 is a reasonable starting point; watch val loss for early stopping.
# batch_size=32 works on CPU; reduce to 16 if you run out of memory.

model.train(
    train,
    val,
    epochs=30,
    batch_size=32,
    save_path=os.path.join(MODELS_DIR, "orthoptera_checkpoints"),
    save_interval=5,       # save checkpoint every 5 epochs
    num_workers=2,
)

In [ ]:
# ── Evaluate on held-out test set ─────────────────────────────────────────────
import matplotlib.pyplot as plt

scores = model.predict(test)
print(scores.describe())

# Per-class precision / recall at threshold 0.5
from sklearn.metrics import classification_report
y_true = test.values
y_pred = (scores.values > 0.5).astype(int)
print(classification_report(y_true, y_pred, target_names=list(train.columns), zero_division=0))

In [ ]:
# ── Save final model ──────────────────────────────────────────────────────────
# OpenSoundscape saves as a .model file (a torch pickle).
# This is the file you point insect.py at in BASE.

model_path = os.path.join(MODELS_DIR, "orthoptera_uk.model")
model.save(model_path)
print(f"Model saved: {model_path}")
print()
print("Next step: set 'model_path' in config/settings.yaml under 'insect:'")
print("and activate 'insect' in classifiers.active")